# Week 4 — Noise Modelling & Higher-$p$ Analysis

**Project:** Diarka Quantum — QAOA Portfolio Optimiser  
**Goal:** Explain the noiseless→hardware gap from Week 2, not just report it.

Week 2 measured: noiseless approximation ratio **0.942** vs IBM Heron r2 **0.910** — a 3.4pp gap, plus more unique bitstrings on hardware (208→254). This notebook:

1. **Reproduces** the gap in a synthetic Heron r2 noisy simulator.
2. **Decomposes** it — turns on 1q gate → 2q gate → T1/T2 → readout one at a time and attributes the drop to each.
3. **Finds the depth crossover** — QAOA at $p=1,2,3$ noiseless vs noisy.
4. **Optionally runs $p=2$ on real hardware** to anchor the crossover with one real point.

It reuses your Week 2–3 pipeline. The only new module is `src/noise_model.py`, and `src/qaoa.py` gained backward-compatible `estimator=` / `sampler=` / `basis_gates=` hooks (every prior call still works unchanged).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt

# Your existing QAOA pipeline (now with optional noisy-primitive hooks)
from qaoa import optimise_qaoa, sample, approximation_ratio, ground_state_probability

# New Week 4 module
from noise_model import (
    HeronParams, NoiseChannels, synthetic_heron_noise_model,
    noisy_estimator, noisy_sampler, attribution_suite,
    noise_model_from_backend, HERON_BASIS_GATES,
)

RNG_SEED = 42
SHOTS = 4096
np.random.seed(RNG_SEED)
os.makedirs('../figures', exist_ok=True)
os.makedirs('../results', exist_ok=True)
print('Imports OK')

## 1. Rebuild the same problem as Weeks 2–3

**⬇️ This is the one cell to wire to your own code.** Paste the Hamiltonian-construction lines from your `06_robustness_and_sensitivity.ipynb` (data → QUBO → Ising) so the result is comparable to prior weeks. The cell must end with a `SparsePauliOp` named `ising` and the same risk-aversion `q` you used for the baseline.

Everything downstream derives from `ising` alone — no other assumptions about your `data`/`classical`/`encoding` signatures.

In [ ]:
# --- PASTE YOUR WEEK 2/3 SETUP HERE, ending with `ising` (SparsePauliOp). ---
# Example shape (replace with your actual calls / cached data):
#
#   from data import fetch_prices, compute_log_returns, annualise
#   from encoding import build_qubo, qubo_to_ising
#   stats = annualise(compute_log_returns(fetch_prices(tickers, start, end)))
#   qubo  = build_qubo(stats, risk_aversion=0.5, budget=4)
#   ising = qubo_to_ising(qubo)
#
# If you cached the Ising from Week 2/3, you can also just reload it.

ising = None  # <-- must be a SparsePauliOp after your setup runs
assert ising is not None, 'Paste your Week 2/3 Hamiltonian setup so `ising` is defined.'
n_qubits = ising.num_qubits
print(f'Hamiltonian on {n_qubits} qubits, {len(ising)} terms')

In [ ]:
# Spectrum bounds + ground bitstring, derived directly from `ising`.
# The QAOA cost Hamiltonian is DIAGONAL (only I/Z terms), so every eigenstate
# is a computational basis state and we can read the spectrum off the diagonal
# without depending on any external eigensolver helper.
diag = np.real(np.diag(ising.to_matrix()))
ground_energy = float(diag.min())
max_energy = float(diag.max())
ground_idx = int(np.argmin(diag))
# Qiskit measure_all keys are q_{n-1}...q_0 (qubit 0 is the LSB), which is exactly
# the binary expansion of the matrix index. So this string matches counts keys.
ground_bitstring = format(ground_idx, f'0{n_qubits}b')
print(f'Ground energy: {ground_energy:.4f}  bitstring: {ground_bitstring}')
print(f'Max energy:    {max_energy:.4f}')

## 2. Noiseless baseline (statevector, your default path)

In [ ]:
res_ideal = optimise_qaoa(ising, p=1)                     # default StatevectorEstimator
res_ideal = sample(res_ideal, shots=SHOTS, seed=RNG_SEED) # default StatevectorSampler

r_ideal = approximation_ratio(res_ideal.optimal_energy,
                              ground_energy=ground_energy, max_energy=max_energy)
pgs_ideal = ground_state_probability(res_ideal.counts, ground_bitstring, res_ideal.shots)
print(f'noiseless: ratio={r_ideal:.4f}  P(GS)={pgs_ideal*100:.3f}%  unique={len(res_ideal.counts)}')

## 3. Reproduce the gap (full synthetic Heron r2 noise at $p=1$)

We reuse the noiseless optimal angles rather than re-optimising under noise — this isolates the *measurement* effect of noise and mirrors how you'd run fixed angles on hardware. Target: noisy ratio near the measured **0.910**.

In [ ]:
nm_full = synthetic_heron_noise_model(n_qubits=n_qubits)
print('Noise registered for:', sorted(nm_full.noise_instructions))

res_noisy = optimise_qaoa(
    ising, p=1,
    estimator=noisy_estimator(nm_full, seed=RNG_SEED),
    basis_gates=HERON_BASIS_GATES,
    initial_params=res_ideal.optimal_params,   # reuse noiseless optimum
    maxiter=1,                                  # evaluate at fixed angles, don't re-optimise
)
res_noisy = sample(res_noisy, shots=SHOTS, sampler=noisy_sampler(nm_full, seed=RNG_SEED))

r_noisy = approximation_ratio(res_noisy.optimal_energy,
                              ground_energy=ground_energy, max_energy=max_energy)
pgs_noisy = ground_state_probability(res_noisy.counts, ground_bitstring, res_noisy.shots)
cz_count = res_noisy.ansatz.count_ops().get('cz', 0)
print(f'noisy:     ratio={r_noisy:.4f}  P(GS)={pgs_noisy*100:.3f}%  unique={len(res_noisy.counts)}')
print(f'CZ gates in transpiled p=1 circuit: {cz_count}')
print(f'simulated gap: {(r_ideal - r_noisy)*100:.2f} pp   (measured hardware gap was 3.40 pp)')

If the simulated gap is far from 3.4pp, tune `HeronParams` in `src/noise_model.py` — most sensitive knobs are `two_qubit_gate_error` and `readout_error` — and re-run. Matching the measured gap is itself a credible, quotable result.

## 4. Noise decomposition — the headline plot

Accumulate channels one at a time. The drop introduced at each step is that channel's contribution to the total gap.

In [ ]:
labels, ratios = [], []
for name, ch in attribution_suite().items():
    nm = synthetic_heron_noise_model(channels=ch, n_qubits=n_qubits)
    res = optimise_qaoa(
        ising, p=1,
        estimator=noisy_estimator(nm, seed=RNG_SEED),
        basis_gates=HERON_BASIS_GATES,
        initial_params=res_ideal.optimal_params,
        maxiter=1,
    )
    r = approximation_ratio(res.optimal_energy, ground_energy=ground_energy, max_energy=max_energy)
    labels.append(name); ratios.append(r)
    print(f'{name:18s} ratio={r:.4f}')

contributions = [ratios[i-1] - ratios[i] for i in range(1, len(ratios))]
contrib_labels = labels[1:]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(labels, ratios, 'o-', color='#2dd4bf', lw=2, ms=8)
ax1.axhline(0.910, ls='--', color='#f0b429', label='measured hardware (0.910)')
ax1.set_ylabel('Approximation ratio'); ax1.set_title('Ratio vs accumulated noise')
ax1.tick_params(axis='x', rotation=30); ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(contrib_labels, [c*100 for c in contributions], color='#7c6cf0')
ax2.set_ylabel('Contribution to gap (pp)'); ax2.set_title('Where the noise comes from')
ax2.tick_params(axis='x', rotation=30); ax2.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../figures/week4_noise_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/week4_noise_decomposition.png')

## 5. Depth crossover — $p=1,2,3$ noiseless vs noisy

Noiselessly, more layers help monotonically. Under noise, each layer roughly doubles the CZ count, so noise eventually wins. The crossover is the core Week 4 physics insight.

In [ ]:
p_values = [1, 2, 3]
ideal_by_p, noisy_by_p, cz_by_p = [], [], []

for p in p_values:
    ri = optimise_qaoa(ising, p=p)                       # full noiseless optimisation
    r_i = approximation_ratio(ri.optimal_energy, ground_energy=ground_energy, max_energy=max_energy)

    rn = optimise_qaoa(ising, p=p,
                       estimator=noisy_estimator(synthetic_heron_noise_model(n_qubits=n_qubits), seed=RNG_SEED),
                       basis_gates=HERON_BASIS_GATES,
                       initial_params=ri.optimal_params, maxiter=1)  # fixed noiseless angles
    r_n = approximation_ratio(rn.optimal_energy, ground_energy=ground_energy, max_energy=max_energy)

    ideal_by_p.append(r_i); noisy_by_p.append(r_n)
    cz_by_p.append(rn.ansatz.count_ops().get('cz', 0))
    print(f'p={p}: noiseless={r_i:.4f}  noisy={r_n:.4f}  (CZ={cz_by_p[-1]})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_values, ideal_by_p, 'o-', color='#2dd4bf', lw=2, ms=9, label='noiseless')
ax.plot(p_values, noisy_by_p, 's--', color='#f0b429', lw=2, ms=9, label='noisy (Heron model)')
for p, r, cz in zip(p_values, noisy_by_p, cz_by_p):
    ax.annotate(f'{cz} CZ', (p, r), textcoords='offset points', xytext=(0, -16), fontsize=8, ha='center')
ax.set_xlabel('QAOA depth $p$'); ax.set_ylabel('Approximation ratio')
ax.set_title('Depth helps noiselessly, hurts under noise')
ax.set_xticks(p_values); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/week4_depth_crossover.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/week4_depth_crossover.png')

## 6. Hardware run at $p=2$ (spends IBM Quantum quota)

**Leave `RUN_HARDWARE = False` until everything above runs.** This submits ONE $p=2$ job at the noiseless-optimal angles — do not re-optimise on hardware. For the noisy-simulator transpile we use `basis_gates`; for real hardware we transpile against the backend so the layout/coupling map is respected.

In [ ]:
RUN_HARDWARE = False

if RUN_HARDWARE:
    from qiskit import transpile
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler
    from qaoa import qaoa_ansatz

    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=n_qubits)
    print('Backend:', backend.name)

    # Noiseless-optimal p=2 angles
    ri_p2 = optimise_qaoa(ising, p=2)

    # Transpile against the REAL backend (respects coupling map + native gates)
    ansatz = qaoa_ansatz(ising, p=2)
    ansatz.measure_all()
    isa = transpile(ansatz, backend=backend, optimization_level=3)
    bound = isa.assign_parameters(ri_p2.optimal_params)

    sampler = RuntimeSampler(mode=backend)
    job = sampler.run([bound], shots=SHOTS)
    print('Job ID:', job.job_id(), '— save this for your portfolio log')
    counts = job.result()[0].data.meas.get_counts()

    # Energy from measured distribution (diagonal H lookup over the sample)
    idx = {format(i, f'0{n_qubits}b'): diag[i] for i in range(2**n_qubits)}
    e_hw = sum(idx[b]*c for b, c in counts.items()) / sum(counts.values())
    r_hw_p2 = approximation_ratio(e_hw, ground_energy=ground_energy, max_energy=max_energy)
    pgs_hw = ground_state_probability(counts, ground_bitstring, sum(counts.values()))
    print(f'HARDWARE p=2: ratio={r_hw_p2:.4f}  P(GS)={pgs_hw*100:.3f}%  unique={len(counts)}')
else:
    print('Hardware run skipped. Set RUN_HARDWARE = True when ready.')

## 7. Persist results for the writeup

In [ ]:
import json
summary = {
    'p1_ratio_ideal': float(r_ideal),
    'p1_ratio_noisy': float(r_noisy),
    'p1_pgs_ideal': float(pgs_ideal),
    'p1_pgs_noisy': float(pgs_noisy),
    'simulated_gap_pp': float((r_ideal - r_noisy) * 100),
    'cz_count_p1': int(cz_count),
    'decomposition_pp': {contrib_labels[i]: float(contributions[i]*100) for i in range(len(contributions))},
    'ideal_by_p': {str(p): float(v) for p, v in zip(p_values, ideal_by_p)},
    'noisy_by_p': {str(p): float(v) for p, v in zip(p_values, noisy_by_p)},
    'cz_by_p': {str(p): int(v) for p, v in zip(p_values, cz_by_p)},
}
with open('../results/week4_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))